# Data Assimilation & The Kalman Gain

Welcome to the interactive notebook for data assimilation and the Kalman gain of the `digital-twins` repository.

In Data Assimilation, we constantly face a dilemma: our **Simulation Model** has errors (Process Noise), but our **Sensor Measurements** also have errors (Sensor Noise). How do we combine them to find the true state?

The **Kalman Filter** solves this elegantly using the **Kalman Gain ($K$)**. The Kalman Gain acts as a relative weight that decides how much we should trust the Sensor versus our Model's Prediction (the Prior).

### The Kalman Gain as a Ratio of Uncertainties

In a 1D space, the Kalman Gain is calculated as:

$$ K = \frac{\Sigma_{prior}}{\Sigma_{prior} + R} $$

Where:
- **$\Sigma_{prior}$** is our Model's Uncertainty (variance based on process noise).
- **$R$** is our Sensor's Uncertainty (variance based on hardware precision).

**The Rule of Thumb:**
The updated belief (the **Posterior**) will always shift closer to the source of information that has the *lowest variance* (the narrowest curve). 
- If the sensor is highly precise ($R \to 0$), $K \to 1$, and we trust the sensor.
- If the model is highly certain ($\Sigma_{prior} \to 0$), $K \to 0$, and we ignore the sensor.

### Interactive Learning
Use the sliders below to adjust the Model Uncertainty and the Sensor Uncertainty. Watch how the Kalman Gain automatically adjusts, causing the solid green **Posterior** curve to slide toward whichever source you make more "trustworthy" (lower variance).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from ipywidgets import interact, FloatSlider

In [ ]:
def interactive_kalman_gain(prior_variance, sensor_variance):
    # 1. Set Constants
    prior_mean = 3.0           # Where the simulation model predicts the object is
    sensor_measurement = 7.0   # Where the sensor actually measured the object
    x = np.linspace(-2, 12, 1000)  # X-axis space

    # 2. Calculate the Math (1D Kalman Filter Update)
    # Calculate the Kalman Gain (K)
    K = prior_variance / (prior_variance + sensor_variance)
    
    # Calculate the new Posterior Mean and Variance
    posterior_mean = prior_mean + K * (sensor_measurement - prior_mean)
    posterior_variance = (1.0 - K) * prior_variance

    # 3. Generate the Gaussian Distributions (PDFs)
    prior_pdf = stats.norm.pdf(x, prior_mean, np.sqrt(prior_variance))
    sensor_pdf = stats.norm.pdf(x, sensor_measurement, np.sqrt(sensor_variance))
    posterior_pdf = stats.norm.pdf(x, posterior_mean, np.sqrt(posterior_variance))

    # 4. Plotting
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Plot Prior (Model Prediction)
    ax.plot(x, prior_pdf, 'b--', label="Model Prediction (Prior)", linewidth=2)
    ax.fill_between(x, prior_pdf, alpha=0.1, color='blue')
    ax.axvline(prior_mean, color='blue', linestyle=':', alpha=0.6)

    # Plot Likelihood (Sensor Measurement)
    ax.plot(x, sensor_pdf, 'r--', label="Sensor Measurement", linewidth=2)
    ax.fill_between(x, sensor_pdf, alpha=0.1, color='red')
    ax.axvline(sensor_measurement, color='red', linestyle=':', alpha=0.6)

    # Plot Posterior (Updated Belief)
    ax.plot(x, posterior_pdf, 'g-', label="Updated Belief (Posterior)", linewidth=3)
    ax.fill_between(x, posterior_pdf, alpha=0.4, color='green')
    ax.axvline(posterior_mean, color='green', linestyle=':', alpha=0.8)

    # Styling
    ax.set_xlim(-2, 12)
    # Dynamically scale Y-axis to fit the tallest peak nicely
    max_y = max(np.max(prior_pdf), np.max(sensor_pdf), np.max(posterior_pdf))
    ax.set_ylim(0, max_y * 1.15)
    
    ax.set_title(f"1D Kalman Update | Current Kalman Gain (K) = {K:.3f}", fontsize=16, fontweight='bold')
    ax.set_xlabel("1D State Space (Position)", fontsize=12)
    ax.set_ylabel("Probability Density", fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right', fontsize=11)
    
    plt.tight_layout()
    plt.show()

# 5. Create the ipywidgets Interaction
interact(interactive_kalman_gain, 
         prior_variance=FloatSlider(value=2.0, min=0.1, max=10.0, step=0.1, description='Model Noise ($\Sigma$):', layout={'width': '500px'}),
         sensor_variance=FloatSlider(value=2.0, min=0.1, max=10.0, step=0.1, description='Sensor Noise ($R$):', layout={'width': '500px'}));